## Agent的基本使用

### 1、创建Agent

In [24]:
# model
from rich import print as rprint
from dotenv import load_dotenv
load_dotenv(override=True)
from langchain.chat_models import init_chat_model
model = init_chat_model(
    model="deepseek-v4-flash", # 模型名称
    extra_body={
        "thinking": {"type": "disabled"}
    }
)

In [6]:
# tool

# 自定义工具
from langchain_core.tools import tool
@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """
    获取指定城市的信息

    Args:
         city : 具体的城市

    Returns:
        返回城市的天气信息
    """
    return city + "晴天，温度15°C"

# 使用langchain工具
from langchain_tavily import TavilySearch

tavily_search_tool = TavilySearch(
    max_results=1,
    topic="general",
)


In [16]:
# agent
from langchain.agents import create_agent

agent = create_agent(
    name="test_agent", # 一般用于Multi-Agent场景
    model = model,
    tools = [get_weather, tavily_search_tool],
    # system_prompt="你是一个话少的助手"
)

rprint(agent)
rprint(type(agent))

<langgraph.graph.state.CompiledStateGraph object at 0x00000149B71D0D60>

<class 'langgraph.graph.state.CompiledStateGraph'>

### 2、agent调用

In [18]:
# 入参类型为字典，输出类型为字典
response = agent.invoke({
    "messages": [
        {"role": "system", "content": "你是一个话少的助手"},
        {"role": "user", "content": "北京的天气如何"},
        {"role": "user", "content": "2026年巴威台风对苏州的影响如何"}
    ]
})

rprint(response)

{
    'messages': [
        SystemMessage(
            content='你是一个话少的助手',
            additional_kwargs={},
            response_metadata={},
            id='7e56f8de-372d-484d-8d26-7b09787bf62c'
        ),
        HumanMessage(
            content='北京的天气如何',
            additional_kwargs={},
            response_metadata={},
            id='6be8e471-ddfb-4e1a-8577-d624ef821ce8'
        ),
        HumanMessage(
            content='2026年巴威台风对苏州的影响如何',
            additional_kwargs={},
            response_metadata={},
            id='aa955231-2106-4527-9a7b-0ed0a3a6a9c7'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': 
'用户问了两个问题。第一个是北京的天气，第二个是2026年巴威台风对苏州的影响。\n\n关于第一个问题，我可以用get_weather
工具查询北京的天气。\n\n关于第二个问题，"2026年巴威台风" - 
这可能是一个未来的台风，或者用户可能指的是过去某个台风？但用户明确说"2026年巴威台风"，这可能是一个预测或者是一个假
设性问题。不过，我也可以搜索一下相关信息。\n\n让我先查北京的天气，同时搜索一下2026年巴威台风的信息。\n\n等等，"巴威
台风" - 
2020年确实有一个台风叫"巴威"（Bavi）。但用户说的是"2026年"，可能是用户搞错了年份，或者确实有关于2026年台风的预测？
我先搜索看看。'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 246,
                    'prompt_tokens': 1882,
                    'total_tokens': 2128,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 159,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 1792},
                    'prompt_cache_hit_tokens': 1792,
                    'prompt_cache_miss_tokens': 90
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                'id': '8b29cb1b-744c-41d3-9e33-7812e20d04fc',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            name='test_agent',
            id='lc_run--019f83a8-4cdc-7321-84bb-81d79ed6144b-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'id': 'call_00_6W2D6Q9JqSUKU8KYhuzG3049',
                    'type': 'tool_call'
                },
                {
                    'name': 'tavily_search',
                    'args': {'query': '2026年 巴威台风 苏州 影响'},
                    'id': 'call_01_9dvo03208GejT8QF4QwI7967',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1882,
                'output_tokens': 246,
                'total_tokens': 2128,
                'input_token_details': {'cache_read': 1792},
                'output_token_details': {'reasoning': 159}
            }
        ),
        ToolMessage(
            content='北京晴天，温度15°C',
            name='get_weather',
            id='1a27ca40-cc24-4e6b-862c-0f535cd869a3',
            tool_call_id='call_00_6W2D6Q9JqSUKU8KYhuzG3049'
        ),
        ToolMessage(
            content='{"query": "2026年 巴威台风 苏州 影响", "follow_up_questions": null, "answer": null, "images": 
[], "results": [{"url": "https://finance.sina.com.cn/wm/2026-07-11/doc-inihmrew3330444.shtml?froms=ggmp", "title": 
"台风“巴威”对苏州影响预计增大", "content": 
"苏州气象消息，台风“巴威”外围雨带已影响苏州，因路径调整，其对苏州影响增强。11 - 
13日有持续8级以上东南大风，12号风力最大，阵风11 - 12级，14号风力渐", "score": 0.90830135, "raw_content": null}], 
"response_time": 0.0, "request_id": "53ed47a3-796e-4f06-9f4d-de3676a041d4"}',
            name='tavily_search',
            id='c97239e8-ce6a-484e-8796-9a4c34a28974',
            tool_call_id='call_01_9dvo03208GejT8QF4QwI7967'
        ),
        AIMessage(
            content='**北京天气**：☀️ 
晴天，15°C。\n\n**2026年巴威台风对苏州的影响**：根据气象信息，台风"巴威"外围雨带已影响苏州，

## agent结构化输出策略

In [25]:
# Pydantic
from pydantic import BaseModel, Field

class ContractInfo(BaseModel):
    name: str = Field(description="用户姓名")
    age: int = Field(description="用户年龄")
    address: str = Field(description="用户地址")


In [26]:
# agent
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy

agent = create_agent(
    name="structured_output_agent", # 一般用于Multi-Agent场景
    model = model,
    system_prompt="你是一个话少的助手",
    response_format = ToolStrategy(ContractInfo)
)

response = agent.invoke({
    "messages": [
        {"role": "user", "content": "有一个客户名字叫Alice，她是个18岁的女生，她和父母住在一起，在北京的东城区"},
    ]
})

rprint(response)

{
    'messages': [
        HumanMessage(
            content='有一个客户名字叫Alice，她是个18岁的女生，她和父母住在一起，在北京的东城区',
            additional_kwargs={},
            response_metadata={},
            id='4962bc5f-7058-49bc-90f0-3f108890e936'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 70,
                    'prompt_tokens': 344,
                    'total_tokens': 414,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 344
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                'id': '6d471655-2db9-46b1-8c8c-a2446556dabb',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            name='structured_output_agent',
            id='lc_run--019f8460-ad7c-78e2-8198-6fdeab213a57-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': 'Alice', 'age': 18, 'address': '北京市东城区'},
                    'id': 'call_00_VVZluNY3a4iwbqPHyDD33054',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 344,
                'output_tokens': 70,
                'total_tokens': 414,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='Alice' age=18 address='北京市东城区'",
            name='ContractInfo',
            id='cb1a0d80-55da-4445-b3c1-23ef4a8e6027',
            tool_call_id='call_00_VVZluNY3a4iwbqPHyDD33054'
        )
    ],
    'structured_response': ContractInfo(name='Alice', age=18, address='北京市东城区')
}